# Experiment overview (non-DTI)

Set the runs root and (optionally) a list of run names to include. The cell below will summarize counts and hygiene metrics and emit a LaTeX table.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from experiments.common import find_runs, load_summary, format_latex_table

# Configure here
run_root = Path("runs")  # change to your runs directory
selected_runs = None      # e.g., ["tdc_caco2_wang_scaffold", "tdc_cyp1a2_veith_random"]; None = all

runs = find_runs(run_root, include_dti=False)
if selected_runs:
    runs = [r for r in runs if r.name in set(selected_runs)]

rows = []
for run in runs:
    summary = load_summary(run)
    counts = summary.get("counts", {})
    hygiene = summary.get("hygiene", {})
    conflicts = summary.get("conflicts", {})
    cliffs = summary.get("cliffs", {})
    rows.append(
        {
            "Dataset": run.name,
            "Train": counts.get("train"),
            "Valid": counts.get("valid"),
            "Test": counts.get("test"),
            "Unique SMILES": hygiene.get("n_unique_valid_smiles"),
            "Duplicates": hygiene.get("n_duplicate_valid_smiles"),
            "Contaminated (tv→test)": hygiene.get("n_contaminated_tv_vs_test"),
            "Conflicts (tv↔test ≥3σ)": conflicts.get("severe_trainvalid_test"),
            "Cliffs (train/valid)": cliffs.get("intra_train_valid"),
            "Cliffs (cross tv↔test)": cliffs.get("cross_tv_test"),
        }
    )

df = pd.DataFrame(rows)
display(df)

latex = format_latex_table(df)
print("\nLaTeX table:\n")
print(latex)
